In [1]:
# ======================================================
# RETAILMART DATASET GENERATOR
# Generates:
#   - retailmart_transactions.csv (1M rows)
#   - customer_feedback.json (250K records)
# ======================================================

# Install required libraries
!pip install -q faker

import pandas as pd
import numpy as np
import json
import uuid
import random
from datetime import datetime, timedelta
from faker import Faker

# ==================== CONFIGURATION ====================
NUM_TRANSACTIONS = 1_000_000
NUM_FEEDBACK = 250_000

# Set seeds for reproducibility
np.random.seed(42)
random.seed(42)
fake = Faker()
Faker.seed(42)

# ==================== REFERENCE DATA ====================

PRODUCT_CATEGORIES = {
    'Electronics': ['Smartphone', 'Laptop', 'Tablet', 'Headphones', 'Smartwatch', 'TV', 'Camera'],
    'Clothing': ['T-Shirt', 'Jeans', 'Jacket', 'Shoes', 'Dress', 'Sweater', 'Shorts'],
    'Home & Kitchen': ['Blender', 'Coffee Maker', 'Cookware Set', 'Toaster', 'Vacuum', 'Dinnerware'],
    'Groceries': ['Milk', 'Bread', 'Eggs', 'Rice', 'Pasta', 'Cereal', 'Snacks'],
    'Beauty': ['Shampoo', 'Moisturizer', 'Perfume', 'Makeup Kit', 'Sunscreen', 'Lipstick'],
    'Sports': ['Yoga Mat', 'Dumbbells', 'Running Shoes', 'Bicycle', 'Tennis Racket', 'Backpack'],
    'Toys': ['Action Figure', 'Board Game', 'Puzzle', 'Doll', 'Building Blocks', 'Remote Car'],
    'Books': ['Novel', 'Textbook', 'Cookbook', 'Biography', 'Comic', 'Self-Help'],
}

STORE_LOCATIONS = ['New York, NY', 'Los Angeles, CA', 'Chicago, IL', 'Houston, TX', 'Phoenix, AZ',
                   'Philadelphia, PA', 'San Antonio, TX', 'San Diego, CA', 'Dallas, TX', 'Austin, TX',
                   'San Francisco, CA', 'Boston, MA', 'Seattle, WA', 'Denver, CO', 'Miami, FL',
                   'Atlanta, GA', 'Portland, OR', 'Las Vegas, NV', 'Nashville, TN', 'Minneapolis, MN']

PAYMENT_METHODS = ['Credit Card', 'Debit Card', 'Cash', 'Mobile Pay', 'Gift Card']
GENDERS = ['Male', 'Female', 'Non-Binary']
CHURN_STATUSES = ['Yes', 'No']
FEEDBACK_PLATFORMS = ['Web', 'Mobile App', 'In-Store', 'Email', 'Phone']

POSITIVE_FEEDBACK = [
    "Great products and fast delivery!",
    "Excellent customer service, very helpful.",
    "Love shopping here, always find what I need.",
    "Fantastic experience, will shop again.",
    "Best store ever, highly recommend!",
    "Amazing quality and great prices.",
    "Super fast shipping and great packaging.",
    "Very satisfied with my purchase.",
    "The staff was friendly and knowledgeable.",
    "Perfect shopping experience from start to finish.",
]

NEUTRAL_FEEDBACK = [
    "Products are okay, nothing special.",
    "Standard experience, no complaints.",
    "Decent selection, average prices.",
    "It was fine, nothing remarkable.",
    "Just a regular shopping trip.",
    "Products are as expected.",
    "No issues, but no wow factor either.",
    "Okay experience overall.",
    "Average store, average service.",
    "It works for basic needs.",
]

NEGATIVE_FEEDBACK = [
    "Poor customer service, very disappointed.",
    "Products arrived damaged, terrible packaging.",
    "Long wait times, won't shop here again.",
    "Overpriced and poor quality.",
    "Rude staff and bad experience.",
    "Never received my order, very frustrated.",
    "Products not as described, misleading.",
    "Worst shopping experience ever.",
    "Terrible return policy, very unhelpful.",
    "Would not recommend this store to anyone.",
]

# ==================== TRANSACTION GENERATION ====================

print(f"Generating {NUM_TRANSACTIONS:,} transaction records...")

def generate_transaction(transaction_id):
    """Generate a single transaction record."""

    # Customer demographics
    customer_age = int(np.random.normal(42, 15))
    customer_age = max(18, min(85, customer_age))
    customer_gender = random.choice(GENDERS)

    # Loyalty and visits
    loyalty_years = int(np.random.exponential(3))
    loyalty_years = min(loyalty_years, 10)
    num_visits = int(np.random.poisson(5 + loyalty_years * 2))
    num_visits = max(1, min(num_visits, 100))

    # Product selection
    category = random.choice(list(PRODUCT_CATEGORIES.keys()))
    product_name = random.choice(PRODUCT_CATEGORIES[category])

    # Pricing based on category
    base_prices = {
        'Electronics': 250,
        'Clothing': 45,
        'Home & Kitchen': 85,
        'Groceries': 12,
        'Beauty': 25,
        'Sports': 75,
        'Toys': 35,
        'Books': 18,
    }
    base_price = base_prices.get(category, 50)
    unit_price = round(max(5.0, np.random.normal(base_price, base_price * 0.3)), 2)

    # Quantity
    quantity = int(np.random.exponential(2)) + 1
    quantity = min(quantity, 20)

    # Total amount
    total_amount = round(unit_price * quantity, 2)

    # Store location
    store_location = random.choice(STORE_LOCATIONS)

    # Payment method
    payment_method = random.choice(PAYMENT_METHODS)

    # Transaction date
    start_date = datetime(2021, 1, 1)
    end_date = datetime(2026, 6, 30)
    days_diff = (end_date - start_date).days
    random_days = random.randint(0, days_diff)
    transaction_date = start_date + timedelta(days=random_days,
                                               hours=random.randint(8, 21),
                                               minutes=random.randint(0, 59),
                                               seconds=random.randint(0, 59))

    # Churn probability based on behaviour
    churn_prob = 0.1
    if loyalty_years < 1:
        churn_prob += 0.25
    if num_visits < 3:
        churn_prob += 0.20
    if total_amount < 30:
        churn_prob += 0.10
    if customer_age < 25:
        churn_prob += 0.10
    if payment_method == 'Cash':
        churn_prob += 0.05

    churn = 'Yes' if random.random() < min(churn_prob, 0.85) else 'No'

    return {
        'transaction_id': transaction_id,
        'customer_id': str(uuid.uuid4()),
        'transaction_date': transaction_date.strftime('%Y-%m-%d %H:%M:%S'),
        'product_category': category,
        'product_name': product_name,
        'quantity': quantity,
        'unit_price': unit_price,
        'total_amount': total_amount,
        'store_location': store_location,
        'payment_method': payment_method,
        'customer_age': customer_age,
        'customer_gender': customer_gender,
        'loyalty_years': loyalty_years,
        'num_visits': num_visits,
        'churn': churn,
    }

# Generate transactions in chunks
print("Generating transactions...")
transaction_data = []
chunk_size = 100_000

for i in range(NUM_TRANSACTIONS):
    transaction_data.append(generate_transaction(f"TXN-{i+1:08d}"))

    if (i + 1) % chunk_size == 0:
        print(f"  Generated {i + 1:,} transactions...")

# Create DataFrame
df_transactions = pd.DataFrame(transaction_data)
print(f" Generated {len(df_transactions):,} transaction records")

# ======================================================
# INTRODUCE DATA QUALITY ISSUES (for Unit 1 practical)
# ======================================================

print("\n🔧 Introducing data quality issues...")

# NULL VALUES
print("  - Adding null values...")

# Null in customer_gender (4%)
mask = np.random.random(len(df_transactions)) < 0.04
df_transactions.loc[mask, 'customer_gender'] = np.nan

# Null in customer_age (3%)
mask = np.random.random(len(df_transactions)) < 0.03
df_transactions.loc[mask, 'customer_age'] = np.nan

# Null in payment_method (2%)
mask = np.random.random(len(df_transactions)) < 0.02
df_transactions.loc[mask, 'payment_method'] = np.nan

# Null in total_amount (1%)
mask = np.random.random(len(df_transactions)) < 0.01
df_transactions.loc[mask, 'total_amount'] = np.nan

# Null in customer_id (0.5%)
mask = np.random.random(len(df_transactions)) < 0.005
df_transactions.loc[mask, 'customer_id'] = np.nan

#  BLANK STRINGS
print("  - Adding blank strings...")

mask = np.random.random(len(df_transactions)) < 0.01
df_transactions.loc[mask, 'customer_gender'] = ''

mask = np.random.random(len(df_transactions)) < 0.005
df_transactions.loc[mask, 'store_location'] = ''

# DUPLICATE transaction_ids (0.5%)
print("  - Adding duplicate transaction_ids...")
dup_count = int(len(df_transactions) * 0.005)
dup_indices = np.random.choice(len(df_transactions), size=dup_count * 2, replace=False)

new_ids = [f"DUP-{i+1:08d}" for i in range(dup_count)]
for i in range(dup_count):
    df_transactions.loc[dup_indices[i], 'transaction_id'] = new_ids[i]
    df_transactions.loc[dup_indices[i + dup_count], 'transaction_id'] = new_ids[i]

# INVALID DATE FORMATS (2%)
print("  - Adding invalid date formats...")
invalid_dates = [
    '2026/01/15 14:30:00',
    '15-01-2026 09:00:00',
    '01/15/2026',
    '2026-01-15',
    'invalid-date',
    '2026-13-45 25:00:00',
    '2026-01-15 14:30',
    '2026-01-15T14:30:00',
]
mask = np.random.random(len(df_transactions)) < 0.02
date_issue_indices = np.where(mask)[0]
for idx in date_issue_indices:
    df_transactions.loc[idx, 'transaction_date'] = random.choice(invalid_dates)

# NEGATIVE VALUES
print("  - Adding negative values...")

# Negative customer_age (0.3%)
mask = np.random.random(len(df_transactions)) < 0.003
df_transactions.loc[mask, 'customer_age'] = -random.randint(1, 20)

# Negative total_amount (0.3%)
mask = np.random.random(len(df_transactions)) < 0.003
df_transactions.loc[mask, 'total_amount'] = -round(random.uniform(5, 50), 2)

# OUT-OF-RANGE VALUES
print("  - Adding out-of-range values...")

# customer_age > 120 (0.3%)
mask = np.random.random(len(df_transactions)) < 0.003
df_transactions.loc[mask, 'customer_age'] = random.randint(121, 150)

# num_visits > 100 (0.3%)
mask = np.random.random(len(df_transactions)) < 0.003
df_transactions.loc[mask, 'num_visits'] = random.randint(101, 500)

# INCONSISTENT CATEGORICAL VALUES
print("  - Adding inconsistent categorical values...")

# Unexpected gender values (0.5%)
mask = np.random.random(len(df_transactions)) < 0.005
unexpected_genders = ['M', 'F', 'Other', 'X', 'Prefer not to say']
df_transactions.loc[mask, 'customer_gender'] = np.random.choice(unexpected_genders, size=mask.sum())

# Unexpected payment methods (0.5%)
mask = np.random.random(len(df_transactions)) < 0.005
unexpected_payments = ['Bitcoin', 'Cash on Delivery', 'Cheque', 'PayPal', 'Venmo']
df_transactions.loc[mask, 'payment_method'] = np.random.choice(unexpected_payments, size=mask.sum())

# Unexpected churn values (0.3%)
mask = np.random.random(len(df_transactions)) < 0.003
inconsistent_churn = ['Y', 'N', 'TRUE', 'FALSE', '1', '0']
df_transactions.loc[mask, 'churn'] = np.random.choice(inconsistent_churn, size=mask.sum())

# ======================================================
# SAVE TRANSACTIONS
# ======================================================

print("\n Saving transactions to CSV...")
output_csv = 'retailmart_transactions.csv'
df_transactions.to_csv(output_csv, index=False)
print(f" Saved: {output_csv}")
print(f"   - Rows: {len(df_transactions):,}")
print(f"   - Columns: {len(df_transactions.columns)}")
print(f"   - File size: {df_transactions.memory_usage(deep=True).sum() / (1024**2):.2f} MB")

# ==================== FEEDBACK GENERATION ====================

print(f"\nGenerating {NUM_FEEDBACK:,} customer feedback records...")

def generate_feedback():
    """Generate a single feedback record with nested JSON structure."""

    # Sentiment
    sentiment = round(np.random.uniform(-1, 1), 3)

    # Select feedback text based on sentiment
    if sentiment > 0.3:
        feedback_text = random.choice(POSITIVE_FEEDBACK)
        rating = random.randint(4, 5)
    elif sentiment < -0.3:
        feedback_text = random.choice(NEGATIVE_FEEDBACK)
        rating = random.randint(1, 2)
    else:
        feedback_text = random.choice(NEUTRAL_FEEDBACK)
        rating = 3

    # Customer info
    customer_name = fake.name()
    customer_location = random.choice(STORE_LOCATIONS)

    # Product category mentioned
    product_category = random.choice(list(PRODUCT_CATEGORIES.keys()))

    # Store location
    store_location = random.choice(STORE_LOCATIONS)

    # Timestamp
    start_date = datetime(2021, 1, 1)
    end_date = datetime(2026, 6, 30)
    days_diff = (end_date - start_date).days
    random_days = random.randint(0, days_diff)
    timestamp = start_date + timedelta(days=random_days,
                                        hours=random.randint(8, 21),
                                        minutes=random.randint(0, 59))

    # Platform
    platform = random.choice(FEEDBACK_PLATFORMS)

    return {
        'feedback_id': str(uuid.uuid4()),
        'customer': {
            'id': str(uuid.uuid4()),
            'name': customer_name,
            'location': customer_location,
            'age': int(np.random.normal(42, 15)),
            'loyalty_years': int(np.random.exponential(3)),
        },
        'feedback': {
            'text': feedback_text,
            'rating': rating,
            'sentiment': sentiment,
            'product_category': product_category,
            'store_location': store_location,
        },
        'timestamp': timestamp.isoformat(),
        'platform': platform,
        'device_type': random.choice(['iPhone', 'Android', 'Web Browser', 'Tablet']),
    }

# Generate feedback
print("Generating feedback records...")
feedback_data = []
chunk_size = 50_000

for i in range(NUM_FEEDBACK):
    feedback_data.append(generate_feedback())

    if (i + 1) % chunk_size == 0:
        print(f"  Generated {i + 1:,} feedback records...")

# Save as JSON Lines (one JSON object per line)
print("\n Saving feedback to JSON...")
output_json = 'customer_feedback.json'
with open(output_json, 'w') as f:
    for record in feedback_data:
        f.write(json.dumps(record) + '\n')

print(f" Saved: {output_json}")
print(f"   - Records: {len(feedback_data):,}")

# ======================================================
# SUMMARY
# ======================================================

print("\n" + "=" * 70)
print(" RETAILMART DATASETS GENERATED SUCCESSFULLY!")
print("=" * 70)
print(f" {output_csv} ({len(df_transactions):,} rows)")
print(f" {output_json} ({len(feedback_data):,} records)")
print("\n" + "=" * 70)
print(" DATA QUALITY ISSUES INTRODUCED:")
print("=" * 70)
print("""
| Issue Type          | Column(s)          | Approximate % |
|---------------------|--------------------| --------------|
| Null values         | customer_gender    | 4%            |
| Null values         | customer_age       | 3%            |
| Null values         | payment_method     | 2%            |
| Null values         | total_amount       | 1%            |
| Null values         | customer_id        | 0.5%          |
| Blank strings       | customer_gender    | 1%            |
| Blank strings       | store_location     | 0.5%          |
| Duplicate IDs       | transaction_id     | 0.5%          |
| Invalid dates       | transaction_date   | 2%            |
| Negative values     | customer_age       | 0.3%          |
| Negative values     | total_amount       | 0.3%          |
| Out-of-range        | customer_age (>120)| 0.3%          |
| Out-of-range        | num_visits (>100)  | 0.3%          |
| Unexpected values   | customer_gender    | 0.5%          |
| Unexpected values   | payment_method     | 0.5%          |
| Inconsistent values | churn              | 0.3%          |
""")

# ======================================================
# DOWNLOAD INSTRUCTIONS
# ======================================================

print("=" * 70)
print(" TO DOWNLOAD THE DATASETS")
print("=" * 70)
print("""
1. Click the folder icon in the left sidebar of Colab
2. Right-click on 'retailmart_transactions.csv' → Download
3. Right-click on 'customer_feedback.json' → Download
4. Share both files with your group members
""")
print("\n Dataset generation complete!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 35.0 MB/s eta 0:00:00
Generating 1,000,000 transaction records...
Generating transactions...
  Generated 100,000 transactions...
  Generated 200,000 transactions...
  Generated 300,000 transactions...
  Generated 400,000 transactions...
  Generated 500,000 transactions...
  Generated 600,000 transactions...
  Generated 700,000 transactions...
  Generated 800,000 transactions...
  Generated 900,000 transactions...
  Generated 1,000,000 transactions...
 Generated 1,000,000 transaction records

🔧 Introducing data quality issues...
  - Adding null values...
  - Adding blank strings...
  - Adding duplicate transaction_ids...
  - Adding invalid date formats...
  - Adding negative values...
  - Adding out-of-range values...
  - Adding inconsistent categorical values...

 Saving transactions to CSV...
 Saved: retailmart_transactions.csv
   - Rows: 1,000,000
   - Columns: 15
   - File size: 571.89 MB

Generating 250,000 customer feedback re